# Music4All-Onion: Content-Based & Cold-Start Song Discovery in Colab (Item-kNN & Two-Tower)

Uses calendar-year splits (`2014–2017` Train, `2018` Validation, `2019` Test) and trains **Item-kNN Content Similarity** and a **Two-Tower Neural Network** to solve both warm song discovery and cold-start song discovery.

## Notes
1. **Identical Preprocessing & Split Logic:** Uses the exact same chunked `.bz2` stream, SHA-256 `retain_user` filter (`SEED = 42`, `USER_FRACTION = 0.10`), and DuckDB global deduplication as the collaborative filtering notebook.
2. **Multimodal Onion Feature Stack:** Streams and aligns compact feature layers from `id_genres_tf-idf`, `id_tags_tf-idf`, `id_ivec256`, `id_lyrics_word2vec`, `id_lyrics_sentiment_functionals`, fitting `TruncatedSVD` and `StandardScaler` strictly on warm training songs.
3. **Cold-Start Seasonal Lift Projection:** Learns a ridge regression mapping from multimodal song features to training month-of-year popularity lift, allowing brand-new cold songs to receive genre/mood-appropriate seasonal reranking.
4. **Dual-Mode Evaluation:** Evaluates models across **Warm Discovery** (`41,834` warm training songs), **Cold-Start Discovery** (unseen songs), and **Combined Discovery**.


## 1. Install dependencies

Run this cell to install dependencies


In [1]:
%pip install -q "numpy>=2.1" "pandas==2.2.3" "scipy>=1.14" "scikit-learn>=1.6" "duckdb==1.4.1" "pyarrow==19.0.1" "matplotlib==3.9.4" "requests==2.32.4" "tqdm==4.67.1" "threadpoolctl==3.6.0" "torch>=2.2.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.7/384.7 kB 163.6 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 3.0.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
jax 0.11.1 requires scipy>=1.15, but you have scipy 1.13.1 which is incompatible.
torchvision 0.26.0+cu130 requires torch==2.11.0, but you have torch 2.14.1 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is inco

## 2. Configuration

Configuration matches the original ALS notebook (`SEED = 42`, `USER_FRACTION = 0.10`, training `2014–2017`, val `2018`, test `2019`)[cite: 1]. If you ran the ALS notebook in the same Colab session, `CF_RUN_DIR` allows direct reuse of the split manifests and evaluation cohort.


In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"

import bz2
import gc
import hashlib
import importlib.metadata
import json
import math
import re
import shutil
import time
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from requests.adapters import HTTPAdapter
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, normalize
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

SEED = 42
ROOT = Path("/content/music4all_onion")
RUN_NAME = "cbf_yearly_2014_2017_val2018_test2019_users10pct_v1"

# Optional: reuse split manifests & eval cohort from CF run if present
CF_RUN_DIR = ROOT / "runs" / "yearly_2014_2017_val2018_test2019_users10pct_v1"

TRAIN_START_YEAR = 2014
TRAIN_END_YEAR = 2017
VAL_YEAR = 2018
TEST_YEAR = 2019
USER_FRACTION = 0.10
CHUNK_SIZE = 250_000
MIN_TRAIN_ITEMS_PER_USER = 5
MIN_TRAIN_USERS_PER_ITEM = 5
CONFIDENCE_ALPHA = 20.0
DUCKDB_MEMORY = "2GB"
CPU_THREADS = 2
KEEP_STAGING_FILES = False

K = 20
EVAL_USER_LIMIT = 2_000
SCORE_BATCH_SIZE = 64
SEASONAL_SMOOTHING = 100.0
SEASONAL_WEIGHTS = [0.0, 0.25]

# ==============================================================================
# MULTIMODAL CONTENT FEATURE CONFIGURATION
# ==============================================================================
ENABLED_MODALITIES = {
    "genres_tfidf": {
        "file": "id_genres_tf-idf.tsv.bz2",
        "md5": "a742b5fa1d2e2ce780101773e57bb7f5",
        "svd_components": 64,
        "modality_weight": 1.25,
    },
    "tags_tfidf": {
        "file": "id_tags_tf-idf.tsv.bz2",
        "md5": "6681d5c2bf214ad54465286fbbab1ed4",
        "svd_components": 128,
        "modality_weight": 1.25,
    },
    "audio_ivec256": {
        "file": "id_ivec256.tsv.bz2",
        "md5": "ae244a772e1e6711c34d94530e429d6f",
        "svd_components": None,
        "modality_weight": 1.00,
    },
    "lyrics_word2vec": {
        "file": "id_lyrics_word2vec.tsv.bz2",
        "md5": "d0f247bdb645e35955cd40c802ef2af2",
        "svd_components": 64,
        "modality_weight": 0.75,
    },
    "lyrics_sentiment": {
        "file": "id_lyrics_sentiment_functionals.tsv.bz2",
        "md5": "76ee907deda85b10e645731ee496a9e7",
        "svd_components": None,
        "modality_weight": 0.50,
    },
}

ITEM_KNN_GRID = [
    {"top_k_neighbors": 100, "similarity_power": 1.5, "pop_exponent": 0.0},
    {"top_k_neighbors": 100, "similarity_power": 1.5, "pop_exponent": 0.35},
    {"top_k_neighbors": 150, "similarity_power": 2.0, "pop_exponent": 0.50},
]

TWO_TOWER_GRID = [
    {"embed_dim": 128, "hidden_dim": 256, "dropout": 0.15, "temperature": 0.07, "lr": 1e-3, "batch_size": 512},
    {"embed_dim": 128, "hidden_dim": 256, "dropout": 0.20, "temperature": 0.05, "lr": 1e-3, "batch_size": 512},
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RAW_DIR = ROOT / "raw"
RUN_DIR = ROOT / "runs" / RUN_NAME
STAGING_DIR = RUN_DIR / "staging"
SPLIT_DIR = RUN_DIR / "manifests"
MODEL_DATA_DIR = RUN_DIR / "model_data"
REPORT_DIR = RUN_DIR / "reports"
BUNDLE_DIR = RUN_DIR / "bundle"
for directory in (RAW_DIR, RUN_DIR, MODEL_DATA_DIR, REPORT_DIR, BUNDLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)

CONFIG = {
    "seed": SEED, "train_start_year": TRAIN_START_YEAR,
    "train_end_year": TRAIN_END_YEAR, "val_year": VAL_YEAR, "test_year": TEST_YEAR,
    "user_fraction": USER_FRACTION, "chunk_size": CHUNK_SIZE,
    "min_train_items_per_user": MIN_TRAIN_ITEMS_PER_USER,
    "min_train_users_per_item": MIN_TRAIN_USERS_PER_ITEM,
    "k": K, "eval_user_limit": EVAL_USER_LIMIT,
    "enabled_modalities": list(ENABLED_MODALITIES.keys()),
    "device": str(DEVICE),
}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))
print("Compute device:", DEVICE)
print("Local free disk (GiB):", round(shutil.disk_usage(ROOT).free / 2**30, 1))


{
  "seed": 42,
  "train_start_year": 2014,
  "train_end_year": 2017,
  "val_year": 2018,
  "test_year": 2019,
  "user_fraction": 0.1,
  "chunk_size": 250000,
  "min_train_items_per_user": 5,
  "min_train_users_per_item": 5,
  "k": 20,
  "eval_user_limit": 2000,
  "enabled_modalities": [
    "genres_tfidf",
    "tags_tfidf",
    "audio_ivec256",
    "lyrics_word2vec",
    "lyrics_sentiment"
  ],
  "device": "cuda"
}
Compute device: cuda
Local free disk (GiB): 184.3


## 3. Download Interaction & Content Feature Files from Zenodo
The fixed release URL and MD5 below come from the [official v2 file listing](https://zenodo.org/records/15394646). The download is streamed to disk and verified before use; an existing verified file is reused. Interrupted downloads can be restarted by rerunning this cell (the partial download starts over). No archive is expanded into a huge TSV.

We download the interaction data (`userid_trackid_timestamp.tsv.bz2`) and all enabled multimodal feature files. Existing verified files are reused.


In [3]:
RECORD_ID = "15394646"
INTERACTION_FILE = "userid_trackid_timestamp.tsv.bz2"
INTERACTION_MD5 = "dfe82201036765f7463e6f3ce3d0f991"

def file_md5(path):
    digest = hashlib.md5()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def ensure_zenodo_file(filename, expected_md5=None):
    target_path = RAW_DIR / filename
    url = f"https://zenodo.org/records/{RECORD_ID}/files/{filename}?download=1"
    if target_path.exists():
        if expected_md5 and file_md5(target_path) != expected_md5:
            raise ValueError(f"Checksum mismatch for {target_path}. Delete the bad file and rerun.")
        print(f"Reusing verified file: {target_path.name} ({target_path.stat().st_size / 2**20:.1f} MiB)")
        return target_path

    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=Retry(
        total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504]
    )))
    partial = target_path.with_suffix(target_path.suffix + ".part")
    with session.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0)) or None
        with partial.open("wb") as handle, tqdm(
            total=total, unit="B", unit_scale=True, desc=f"Download {filename}"
        ) as progress:
            for block in response.iter_content(chunk_size=8 * 1024 * 1024):
                if block:
                    handle.write(block)
                    progress.update(len(block))
    if expected_md5 and file_md5(partial) != expected_md5:
        raise ValueError(f"Downloaded file {filename} failed MD5 verification.")
    partial.replace(target_path)
    print(f"Verified download: {target_path.name}")
    return target_path

REUSE_CF_MANIFESTS = (
    CF_RUN_DIR.exists()
    and (CF_RUN_DIR / "preprocessing_complete.json").exists()
    and all(list((CF_RUN_DIR / "manifests" / f"split={s}").glob("*.parquet")) for s in ("train", "val", "test"))
)

if REUSE_CF_MANIFESTS and not (RUN_DIR / "preprocessing_complete.json").exists():
    print("Found completed split manifests from Collaborative Filtering run:", CF_RUN_DIR)
else:
    RAW_INTERACTION_PATH = ensure_zenodo_file(INTERACTION_FILE, INTERACTION_MD5)

FEATURE_PATHS = {}
for mod_name, spec in ENABLED_MODALITIES.items():
    FEATURE_PATHS[mod_name] = ensure_zenodo_file(spec["file"], spec.get("md5"))


Reusing verified file: userid_trackid_timestamp.tsv.bz2 (2109.0 MiB)
Reusing verified file: id_genres_tf-idf.tsv.bz2 (2.8 MiB)
Reusing verified file: id_tags_tf-idf.tsv.bz2 (42.9 MiB)
Reusing verified file: id_ivec256.tsv.bz2 (81.8 MiB)
Reusing verified file: id_lyrics_word2vec.tsv.bz2 (232.3 MiB)
Reusing verified file: id_lyrics_sentiment_functionals.tsv.bz2 (29.9 MiB)


## 4. Stream, Clean, Deduplicate Globally, and Split by Calendar Year

Identical to the original ALS notebook preprocessing logic


In [4]:
def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"

if "con" in globals():
    con.close()
con = duckdb.connect(str(RUN_DIR / "cbf_preprocessing.duckdb"))
con.execute(f"SET memory_limit = {sql_string(DUCKDB_MEMORY)}")
con.execute(f"SET threads = {int(CPU_THREADS)}")
con.execute(f"SET temp_directory = {sql_string(RUN_DIR / 'duckdb_tmp')}")
con.execute("SET preserve_insertion_order = false")

ACTIVE_SPLIT_DIR = (CF_RUN_DIR / "manifests") if REUSE_CF_MANIFESTS else SPLIT_DIR

def split_glob(split):
    if split not in ("train", "val", "test"):
        raise ValueError("Expected train, val, or test.")
    return str(ACTIVE_SPLIT_DIR / f"split={split}" / "*.parquet")

def split_scan(split):
    return f"read_parquet({sql_string(split_glob(split))}, hive_partitioning=true)"

@lru_cache(maxsize=500_000)
def retain_user(user_id):
    value = int.from_bytes(
        hashlib.sha256(f"{SEED}:{user_id}".encode("utf-8")).digest()[:8], "big"
    )
    return value < int(USER_FRACTION * 2**64)

def resolve_columns(columns):
    by_normalized = {
        re.sub(r"[^a-z0-9]", "", str(column).lower()): column
        for column in columns
    }
    result = []
    for label, names in (
        ("user", ("userid", "user", "listenerid", "uid")),
        ("song", ("trackid", "itemid", "songid", "musicid", "track", "song")),
        ("time", ("timestamp", "datetime", "playedat", "eventtime", "date")),
    ):
        found = next((by_normalized[name] for name in names if name in by_normalized), None)
        if found is None:
            raise ValueError(f"Cannot resolve {label} column from {list(columns)}.")
        result.append(found)
    return result

completion_path = RUN_DIR / "preprocessing_complete.json"
if REUSE_CF_MANIFESTS:
    cleaning_report = json.loads((CF_RUN_DIR / "preprocessing_complete.json").read_text())
    completion_path.write_text(json.dumps(cleaning_report, indent=2))
    print("Reusing CF split manifests at:", ACTIVE_SPLIT_DIR)
elif completion_path.exists():
    cleaning_report = json.loads(completion_path.read_text())
    print("Reusing completed split manifests at:", ACTIVE_SPLIT_DIR)
else:
    STAGING_DIR.mkdir(parents=True, exist_ok=True)
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    header = pd.read_csv(RAW_INTERACTION_PATH, sep="\t", compression="bz2", nrows=0).columns
    user_col, item_col, time_col = resolve_columns(header)
    cleaning_report = {
        "raw_rows": 0, "invalid_rows": 0, "valid_rows_outside_year_window": 0,
        "in_window_rows_before_sampling": 0, "sampled_rows_before_deduplication": 0,
        "rows_after_chunk_deduplication": 0,
    }
    chunks_written = 0
    reader = pd.read_csv(
        RAW_INTERACTION_PATH, sep="\t", compression="bz2",
        usecols=[user_col, item_col, time_col], dtype="string",
        chunksize=CHUNK_SIZE, keep_default_na=False,
    )
    for chunk in tqdm(reader, desc="Stream compressed events", unit="chunks"):
        cleaning_report["raw_rows"] += len(chunk)
        frame = chunk.rename(columns={
            user_col: "user_id", item_col: "item_id", time_col: "timestamp"
        })[["user_id", "item_id", "timestamp"]].copy()
        frame["user_id"] = frame["user_id"].str.strip()
        frame["item_id"] = frame["item_id"].str.strip()
        frame["timestamp"] = pd.to_datetime(
            frame["timestamp"].str.strip(), format="%Y-%m-%d %H:%M:%S",
            utc=True, errors="coerce",
        )
        valid = (
            frame["timestamp"].notna()
            & frame["user_id"].notna() & frame["user_id"].ne("")
            & frame["item_id"].notna() & frame["item_id"].ne("")
        )
        cleaning_report["invalid_rows"] += int((~valid).sum())
        frame = frame.loc[valid].copy()
        frame["year"] = frame["timestamp"].dt.year.astype("int16")
        frame["month"] = frame["timestamp"].dt.month.astype("int8")
        in_window = frame["year"].between(TRAIN_START_YEAR, TEST_YEAR)
        cleaning_report["valid_rows_outside_year_window"] += int((~in_window).sum())
        frame = frame.loc[in_window].copy()
        cleaning_report["in_window_rows_before_sampling"] += len(frame)
        if USER_FRACTION < 1.0 and not frame.empty:
            sampled = {u: retain_user(u) for u in frame["user_id"].unique()}
            frame = frame.loc[frame["user_id"].map(sampled)].copy()
        cleaning_report["sampled_rows_before_deduplication"] += len(frame)
        frame = frame.drop_duplicates(["user_id", "item_id", "timestamp"])
        cleaning_report["rows_after_chunk_deduplication"] += len(frame)
        if not frame.empty:
            frame.to_parquet(
                STAGING_DIR / f"chunk_{chunks_written:06d}.parquet",
                index=False, compression="zstd",
            )
            chunks_written += 1

    con.execute(f"""
        COPY (
            SELECT DISTINCT user_id, item_id, timestamp, year, month,
                CASE
                    WHEN year <= {TRAIN_END_YEAR} THEN 'train'
                    WHEN year = {VAL_YEAR} THEN 'val'
                    ELSE 'test'
                END AS split
            FROM read_parquet({sql_string(STAGING_DIR / '*.parquet')})
        ) TO {sql_string(SPLIT_DIR)}
        (FORMAT PARQUET, PARTITION_BY (split), COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
    """)
    total_clean = con.execute(f"""
        SELECT count(*) FROM read_parquet(
            {sql_string(SPLIT_DIR / 'split=*' / '*.parquet')},
            hive_partitioning=true
        )
    """).fetchone()[0]
    cleaning_report["deduplicated_events"] = int(total_clean)
    cleaning_report["exact_duplicates_removed"] = (
        cleaning_report["sampled_rows_before_deduplication"] - int(total_clean)
    )
    completion_path.write_text(json.dumps(cleaning_report, indent=2))
    if not KEEP_STAGING_FILES:
        for path in STAGING_DIR.glob("chunk_*.parquet"):
            path.unlink()
        STAGING_DIR.rmdir()

display(pd.Series(cleaning_report, name="rows").to_frame())


Stream compressed events: 0chunks [00:00, ?chunks/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows
raw_rows,252984396
invalid_rows,0
valid_rows_outside_year_window,125060811
in_window_rows_before_sampling,127923585
sampled_rows_before_deduplication,13362394
rows_after_chunk_deduplication,13362376
deduplicated_events,13362376
exact_duplicates_removed,18


## 5. Fit the Training Vocabulary and Expanded Catalog

Applies the exact same training-only frequency thresholds (`MIN_TRAIN_ITEMS_PER_USER = 5`, `MIN_TRAIN_USERS_PER_ITEM = 5`) to construct the warm training vocabulary (`0 .. n_warm_items - 1`), and then registers all remaining split songs as cold-start songs (`n_warm_items .. n_all_items - 1`). Reuses CF evaluation cohort (`eval_user_indices.npy`) if present.


In [5]:
train_scan = split_scan("train")
val_scan = split_scan("val")
test_scan = split_scan("test")

con.execute(f"""
    CREATE OR REPLACE TABLE raw_train_pairs AS
    SELECT user_id, item_id, count(*)::BIGINT AS plays
    FROM {train_scan} GROUP BY user_id, item_id
""")
con.execute(f"""
    CREATE OR REPLACE TABLE retained_users AS
    SELECT user_id FROM raw_train_pairs GROUP BY user_id
    HAVING count(*) >= {MIN_TRAIN_ITEMS_PER_USER}
""")
con.execute(f"""
    CREATE OR REPLACE TABLE retained_items AS
    SELECT p.item_id FROM raw_train_pairs p
    JOIN retained_users u USING (user_id)
    GROUP BY p.item_id HAVING count(*) >= {MIN_TRAIN_USERS_PER_ITEM}
""")
con.execute("""
    CREATE OR REPLACE TABLE filtered_train_pairs AS
    SELECT p.* FROM raw_train_pairs p
    JOIN retained_users u USING (user_id)
    JOIN retained_items i USING (item_id)
""")

con.execute("""
    CREATE OR REPLACE TABLE user_map AS
    SELECT user_id, (row_number() OVER (ORDER BY user_id) - 1)::INTEGER AS user_index
    FROM (SELECT DISTINCT user_id FROM filtered_train_pairs)
""")
con.execute("""
    CREATE OR REPLACE TABLE warm_item_map AS
    SELECT item_id, (row_number() OVER (ORDER BY item_id) - 1)::INTEGER AS item_index,
           true AS is_warm
    FROM (SELECT DISTINCT item_id FROM filtered_train_pairs)
""")

n_users = int(con.execute("SELECT count(*) FROM user_map").fetchone()[0])
n_warm_items = int(con.execute("SELECT count(*) FROM warm_item_map").fetchone()[0])

con.execute(f"""
    CREATE OR REPLACE TABLE all_split_items AS
    SELECT DISTINCT item_id FROM (
        SELECT item_id FROM {train_scan}
        UNION
        SELECT item_id FROM {val_scan}
        UNION
        SELECT item_id FROM {test_scan}
    )
""")

con.execute(f"""
    CREATE OR REPLACE TABLE item_map AS
    SELECT item_id, item_index, is_warm FROM warm_item_map
    UNION ALL
    SELECT a.item_id,
           ({n_warm_items} + row_number() OVER (ORDER BY a.item_id) - 1)::INTEGER AS item_index,
           false AS is_warm
    FROM all_split_items a
    LEFT JOIN warm_item_map w USING (item_id)
    WHERE w.item_id IS NULL
""")

con.execute("""
    CREATE OR REPLACE TABLE train_pairs AS
    SELECT u.user_index, i.item_index, p.plays
    FROM filtered_train_pairs p
    JOIN user_map u USING (user_id)
    JOIN warm_item_map i USING (item_id)
""")

users_df = con.execute("SELECT * FROM user_map ORDER BY user_index").df()
items_df = con.execute("SELECT * FROM item_map ORDER BY item_index").df()
users_df.to_parquet(MODEL_DATA_DIR / "users.parquet", index=False)
items_df.to_parquet(MODEL_DATA_DIR / "items.parquet", index=False)

user_ids = users_df["user_id"].astype(str).to_numpy()
item_ids = items_df["item_id"].astype(str).to_numpy()
item_is_warm = items_df["is_warm"].to_numpy(dtype=bool)
n_all_items = len(item_ids)
n_cold_items = n_all_items - n_warm_items

train_pairs_df = con.execute("SELECT user_index, item_index, plays FROM train_pairs ORDER BY user_index, item_index").df()
train_counts = sparse.csr_matrix(
    (
        train_pairs_df["plays"].to_numpy(dtype=np.float32),
        (
            train_pairs_df["user_index"].to_numpy(dtype=np.int32),
            train_pairs_df["item_index"].to_numpy(dtype=np.int32),
        ),
    ),
    shape=(n_users, n_warm_items),
)
train_counts.sort_indices()
sparse.save_npz(MODEL_DATA_DIR / "train_counts.npz", train_counts)

train_confidence = train_counts.copy()
train_confidence.data = (1.0 + float(CONFIDENCE_ALPHA) * np.log1p(train_confidence.data)).astype(np.float32)

for table in ("raw_train_pairs", "retained_users", "retained_items", "filtered_train_pairs", "all_split_items"):
    con.execute(f"DROP TABLE IF EXISTS {table}")
del train_pairs_df
gc.collect()

cf_cohort_path = CF_RUN_DIR / "model_data" / "eval_user_indices.npy"
if cf_cohort_path.exists():
    full_eval_users = np.load(cf_cohort_path)
    print("Loaded exact evaluation cohort from CF run.")
else:
    rng = np.random.default_rng(SEED)
    if EVAL_USER_LIMIT is None or EVAL_USER_LIMIT >= n_users:
        full_eval_users = np.arange(n_users, dtype=np.int32)
    else:
        full_eval_users = np.sort(rng.choice(n_users, EVAL_USER_LIMIT, replace=False)).astype(np.int32)

np.save(MODEL_DATA_DIR / "eval_user_indices.npy", full_eval_users)
active_eval_users = full_eval_users

con.register("eval_user_ids", pd.DataFrame({"user_index": active_eval_users}))
con.register("full_eval_user_ids", pd.DataFrame({"user_index": full_eval_users}))

print({
    "training_users": n_users, "warm_catalog_songs": n_warm_items,
    "cold_split_songs": n_cold_items, "expanded_catalog_songs": n_all_items,
    "nonzero_training_pairs": int(train_counts.nnz), "eval_cohort_users": len(full_eval_users),
})


{'training_users': 5463, 'warm_catalog_songs': 41834, 'cold_split_songs': 13994, 'expanded_catalog_songs': 55828, 'nonzero_training_pairs': 2116066, 'eval_cohort_users': 2000}


## 6. Multimodal Content Feature Extraction & Leakage-Free Normalization

Streams compressed feature files line-by-line using the RAM-efficient parser (`load_aligned_modality_stream`), fitting `TruncatedSVD` and `StandardScaler` **strictly on warm training songs**. This is done to avoid memory blow-up.


In [6]:
item_id_to_index = {item_id: idx for idx, item_id in enumerate(item_ids)}

def load_aligned_modality_stream(filepath, item_id_to_index, n_total_items, sparse_threshold=512):
    filepath = Path(filepath)
    matched_mask = np.zeros(n_total_items, dtype=bool)

    with bz2.open(filepath, mode="rt", encoding="utf-8", errors="replace") as handle:
        header_line = handle.readline()
        if not header_line:
            raise ValueError(f"Empty feature file: {filepath}")
        n_feats = header_line.count("\t")
        use_sparse = n_feats > sparse_threshold

        if use_sparse:
            row_chunks, col_chunks, val_chunks = [], [], []
            buf_r, buf_c, buf_v = [], [], []
        else:
            aligned = np.zeros((n_total_items, n_feats), dtype=np.float32)

        for line in tqdm(handle, total=109_269, desc=f"Stream {filepath.name}", leave=False):
            tab_pos = line.find("\t")
            if tab_pos == -1:
                continue
            raw_id = line[:tab_pos].strip()
            mapped_idx = item_id_to_index.get(raw_id)
            if mapped_idx is None:
                continue

            vals = np.fromstring(line[tab_pos + 1:], dtype=np.float32, sep="\t")
            if vals.size != n_feats:
                if vals.size < n_feats:
                    padded = np.zeros(n_feats, dtype=np.float32)
                    padded[:vals.size] = vals
                    vals = padded
                else:
                    vals = vals[:n_feats]
            np.nan_to_num(vals, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
            matched_mask[mapped_idx] = True

            if use_sparse:
                nz = np.flatnonzero(vals)
                if nz.size > 0:
                    buf_r.append(np.full(nz.size, mapped_idx, dtype=np.int32))
                    buf_c.append(nz.astype(np.int32))
                    buf_v.append(vals[nz].astype(np.float32))
                    if len(buf_r) >= 4_000:
                        row_chunks.append(np.concatenate(buf_r))
                        col_chunks.append(np.concatenate(buf_c))
                        val_chunks.append(np.concatenate(buf_v))
                        buf_r.clear(); buf_c.clear(); buf_v.clear()
            else:
                aligned[mapped_idx] = vals

    if use_sparse:
        if buf_r:
            row_chunks.append(np.concatenate(buf_r))
            col_chunks.append(np.concatenate(buf_c))
            val_chunks.append(np.concatenate(buf_v))
            buf_r.clear(); buf_c.clear(); buf_v.clear()
        if row_chunks:
            rows = np.concatenate(row_chunks)
            cols = np.concatenate(col_chunks)
            data = np.concatenate(val_chunks)
            del row_chunks, col_chunks, val_chunks
        else:
            rows = np.empty(0, dtype=np.int32)
            cols = np.empty(0, dtype=np.int32)
            data = np.empty(0, dtype=np.float32)
        aligned = sparse.csr_matrix(
            (data, (rows, cols)), shape=(n_total_items, n_feats), dtype=np.float32
        )
        del rows, cols, data
        ram_mib = (aligned.data.nbytes + aligned.indices.nbytes + aligned.indptr.nbytes) / 2**20
        storage_kind = f"sparse CSR ({ram_mib:.1f} MiB)"
    else:
        ram_mib = aligned.nbytes / 2**20
        storage_kind = f"dense float32 ({ram_mib:.1f} MiB)"

    coverage = float(matched_mask.mean())
    print(f"  -> {filepath.name}: raw_dim={n_feats:,}, storage={storage_kind}, catalog_coverage={coverage:.2%}")
    return aligned, matched_mask, n_feats

feature_cache_path = MODEL_DATA_DIR / "multimodal_song_features.npz"
if feature_cache_path.exists():
    cached = np.load(feature_cache_path)
    song_features = cached["song_features"]
    modality_summary = json.loads((MODEL_DATA_DIR / "modality_summary.json").read_text())
    print("Loaded cached multimodal song features:", song_features.shape)
else:
    blocks = []
    modality_summary = []
    for mod_name, spec in ENABLED_MODALITIES.items():
        block_cache_path = MODEL_DATA_DIR / f"block_{mod_name}.npz"
        if block_cache_path.exists():
            b_cached = np.load(block_cache_path)
            weighted_block = b_cached["block"]
            meta = json.loads(str(b_cached["meta"]))
            blocks.append(weighted_block)
            modality_summary.append(meta)
            print(f"  -> Reusing cached block for {mod_name}: shape={weighted_block.shape}")
            continue

        raw_mat, matched, raw_dim = load_aligned_modality_stream(
            FEATURE_PATHS[mod_name], item_id_to_index, n_all_items
        )
        n_svd = spec.get("svd_components")
        if n_svd is not None and raw_dim > n_svd:
            svd = TruncatedSVD(n_components=int(n_svd), random_state=SEED)
            svd.fit(raw_mat[:n_warm_items])  # Fit strictly on warm training catalog
            reduced = svd.transform(raw_mat).astype(np.float32)
            explained = float(np.nan_to_num(svd.explained_variance_ratio_).sum())
            del svd
        else:
            reduced = raw_mat.toarray().astype(np.float32) if sparse.issparse(raw_mat) else raw_mat
            explained = 1.0

        del raw_mat
        gc.collect()

        scaler = StandardScaler()
        scaler.fit(reduced[:n_warm_items])
        standardized = scaler.transform(reduced).astype(np.float32)
        del reduced
        np.clip(standardized, -5.0, 5.0, out=standardized)

        unit_block = normalize(standardized, norm="l2", axis=1)
        del standardized
        weighted_block = (unit_block * np.sqrt(float(spec.get("modality_weight", 1.0)))).astype(np.float32)
        del unit_block

        meta = {
            "modality": mod_name, "raw_dim": int(raw_dim), "encoded_dim": int(weighted_block.shape[1]),
            "warm_coverage": float(matched[:n_warm_items].mean()), "cold_coverage": float(matched[n_warm_items:].mean()) if n_cold_items > 0 else 1.0,
            "svd_explained_var": round(explained, 4), "weight": float(spec.get("modality_weight", 1.0)),
        }
        np.savez_compressed(block_cache_path, block=weighted_block, meta=json.dumps(meta))
        blocks.append(weighted_block)
        modality_summary.append(meta)
        gc.collect()

    song_features = np.concatenate(blocks, axis=1).astype(np.float32)
    del blocks
    gc.collect()
    song_features = normalize(song_features, norm="l2", axis=1).astype(np.float32)
    np.savez_compressed(feature_cache_path, song_features=song_features)
    (MODEL_DATA_DIR / "modality_summary.json").write_text(json.dumps(modality_summary, indent=2))

display(pd.DataFrame(modality_summary))
print("Unified multimodal song feature matrix shape:", song_features.shape, f"({song_features.nbytes / 2**20:.1f} MiB)")


Stream id_genres_tf-idf.tsv.bz2:   0%|          | 0/109269 [00:00<?, ?it/s]

  -> id_genres_tf-idf.tsv.bz2: raw_dim=685, storage=sparse CSR (1.1 MiB), catalog_coverage=100.00%


Stream id_tags_tf-idf.tsv.bz2:   0%|          | 0/109269 [00:00<?, ?it/s]

  -> id_tags_tf-idf.tsv.bz2: raw_dim=178,108, storage=sparse CSR (13.2 MiB), catalog_coverage=81.00%


Stream id_ivec256.tsv.bz2:   0%|          | 0/109269 [00:00<?, ?it/s]

...

  -> id_lyrics_word2vec.tsv.bz2: raw_dim=300, storage=dense float32 (63.9 MiB), catalog_coverage=100.00%


Stream id_lyrics_sentiment_functionals.tsv.bz2:   0%|          | 0/109269 [00:00<?, ?it/s]

  -> id_lyrics_sentiment_functionals.tsv.bz2: raw_dim=64, storage=dense float32 (13.6 MiB), catalog_coverage=100.00%


,modality,raw_dim,encoded_dim,warm_coverage,cold_coverage,svd_explained_var,weight
0,genres_tfidf,685,64,1.000000,1.000000,0.6649,1.25
1,tags_tfidf,178108,128,0.911723,0.505788,0.2050,1.25
2,audio_ivec256,100,100,1.000000,1.000000,1.0000,1.00
3,lyrics_word2vec,300,64,1.000000,1.000000,0.7822,0.75
4,lyrics_sentiment,64,64,1.000000,1.000000,1.0000,0.50


Unified multimodal song feature matrix shape: (55828, 420) (89.4 MiB)


## 7. Train-Only Seasonal Priors & Content-Projected Seasonal Lift for Cold Songs

Computes training-only monthly popularity lift and projects it onto cold-start songs via simple Linear Regression + L2 (Ridge) Regularization. Note that these scores act as post-hoc score adjustments after recommendation model prediction.

Music habits shift throughout the year.

For each warm song (also done in the Collaborative Filtering notebook), we compute a "log-lift" for each month which indicates how much more or less popular a song is for a given month relative to its annual average.

For cold songs (without interaction data), we can't directly do this computation. Instead, we train a ridge regression model on the warm songs, where inputs are a song's multimodal content vectors with the targets being their 12-month seasonal lift curves. We then use this model on the cold songs.


In [17]:
popular = con.execute("""
    SELECT item_index, count(*) AS listeners
    FROM train_pairs GROUP BY item_index
""").df()
global_popularity_warm = np.zeros(n_warm_items, dtype=np.float32)
global_popularity_warm[popular["item_index"].to_numpy()] = popular["listeners"].to_numpy()

month_counts_df = con.execute(f"""
    SELECT e.month, i.item_index, count(DISTINCT e.user_id) AS listeners
    FROM {train_scan} e
    JOIN user_map u USING (user_id)
    JOIN warm_item_map i USING (item_id)
    GROUP BY e.month, i.item_index
""").df()

month_counts = np.zeros((12, n_warm_items), dtype=np.float64)
month_counts[
    month_counts_df["month"].to_numpy(dtype=int) - 1,
    month_counts_df["item_index"].to_numpy(dtype=int),
] = month_counts_df["listeners"].to_numpy()

pooled = month_counts.sum(axis=0)
prior = (pooled + 1.0) / (pooled.sum() + n_warm_items)
seasonal_popularity_warm = (
    month_counts + SEASONAL_SMOOTHING * prior[None, :]
) / (month_counts.sum(axis=1, keepdims=True) + SEASONAL_SMOOTHING)
lift_warm = np.clip(np.log(seasonal_popularity_warm / prior[None, :]), -3.0, 3.0)
seasonal_features_warm = (
    (lift_warm - lift_warm.mean(axis=1, keepdims=True))
    / np.maximum(lift_warm.std(axis=1, keepdims=True), 1e-6)
).astype(np.float32)

seasonal_ridge = Ridge(alpha=100.0, random_state=SEED)
seasonal_ridge.fit(song_features[:n_warm_items], seasonal_features_warm.T)

seasonal_features_all = np.zeros((12, n_all_items), dtype=np.float32)
seasonal_features_all[:, :n_warm_items] = seasonal_features_warm
if n_cold_items > 0:
    cold_pred_seasonal = seasonal_ridge.predict(song_features[n_warm_items:]).T.astype(np.float32)
    cold_pred_seasonal = (
        (cold_pred_seasonal - cold_pred_seasonal.mean(axis=1, keepdims=True))
        / np.maximum(cold_pred_seasonal.std(axis=1, keepdims=True), 1e-6)
    ).astype(np.float32)
    seasonal_features_all[:, n_warm_items:] = cold_pred_seasonal

user_content_profiles = train_confidence @ song_features[:n_warm_items]
user_content_profiles = normalize(user_content_profiles, norm="l2", axis=1).astype(np.float32)

np.savez_compressed(
    MODEL_DATA_DIR / "cbf_priors_and_profiles.npz",
    global_popularity_warm=global_popularity_warm,
    seasonal_features_all=seasonal_features_all,
    user_content_profiles=user_content_profiles,
)
del month_counts, month_counts_df, pooled, prior, lift_warm
gc.collect()
print("Built user content profiles:", user_content_profiles.shape, "and expanded seasonal features:", seasonal_features_all.shape)


Built user content profiles: (5463, 420) and expanded seasonal features: (12, 55828)


## 8. Dual-Mode Evaluation Pipeline (`Warm`, `Cold-Start`, and `Combined` Discovery) Definition

Extracts evaluation queries (`warm`, `cold`, `combined`) and computes ranking metrics (`Precision@K`, `Recall@K`, `NDCG@K`, `HitRate@K`).


In [8]:
def target_queries(split, use_full_cohort=False):
    scan = split_scan(split)
    cohort_table = "full_eval_user_ids" if use_full_cohort else "eval_user_ids"

    joined = f"""
        WITH future AS (
            SELECT DISTINCT month, user_id, item_id FROM {scan}
        )
        SELECT f.month, f.user_id, f.item_id, u.user_index, i.item_index,
               COALESCE(i.is_warm, false) AS is_warm,
               p.user_index IS NOT NULL AS seen_in_train,
               c.user_index IS NOT NULL AS in_eval_cohort
        FROM future f
        LEFT JOIN user_map u USING (user_id)
        LEFT JOIN item_map i USING (item_id)
        LEFT JOIN train_pairs p
            ON p.user_index = u.user_index AND p.item_index = i.item_index
        LEFT JOIN {cohort_table} c ON c.user_index = u.user_index
    """

    coverage = con.execute(f"""
        WITH joined AS ({joined})
        SELECT month,
            count(*) AS all_user_month_song_labels,
            count(DISTINCT user_id) AS all_active_users,
            count(*) FILTER (WHERE user_index IS NULL) AS cold_user_labels,
            count(*) FILTER (WHERE user_index IS NOT NULL AND NOT is_warm) AS cold_item_labels_among_known_users,
            count(*) FILTER (WHERE user_index IS NOT NULL AND is_warm AND seen_in_train) AS warm_previously_seen_labels,
            count(*) FILTER (WHERE user_index IS NOT NULL AND is_warm AND NOT seen_in_train) AS warm_discovery_labels,
            count(*) FILTER (WHERE user_index IS NOT NULL AND is_warm AND NOT seen_in_train AND in_eval_cohort) AS eval_warm_discovery_labels,
            count(*) FILTER (WHERE user_index IS NOT NULL AND NOT is_warm AND in_eval_cohort) AS eval_cold_discovery_labels
        FROM joined GROUP BY month ORDER BY month
    """).df()

    warm_queries = con.execute(f"""
        WITH joined AS ({joined})
        SELECT month, user_index, list(item_index ORDER BY item_index) AS relevant_items
        FROM joined
        WHERE user_index IS NOT NULL AND is_warm AND NOT seen_in_train AND in_eval_cohort
        GROUP BY month, user_index ORDER BY month, user_index
    """).df()

    cold_queries = con.execute(f"""
        WITH joined AS ({joined})
        SELECT month, user_index, list(item_index ORDER BY item_index) AS relevant_items
        FROM joined
        WHERE user_index IS NOT NULL AND NOT is_warm AND item_index IS NOT NULL AND in_eval_cohort
        GROUP BY month, user_index ORDER BY month, user_index
    """).df()

    combined_queries = con.execute(f"""
        WITH joined AS ({joined})
        SELECT month, user_index, list(item_index ORDER BY item_index) AS relevant_items
        FROM joined
        WHERE user_index IS NOT NULL AND item_index IS NOT NULL AND NOT seen_in_train AND in_eval_cohort
        GROUP BY month, user_index ORDER BY month, user_index
    """).df()

    return {"warm": warm_queries, "cold": cold_queries, "combined": combined_queries, "coverage": coverage}

def standardized_rows(values):
    return (
        (values - values.mean(axis=1, keepdims=True))
        / np.maximum(values.std(axis=1, keepdims=True), 1e-6)
    )

METRIC_COLUMNS = [f"precision@{K}", f"recall@{K}", f"ndcg@{K}", f"hitrate@{K}"]

def evaluate_scorer(score_fn, queries_df, label, catalog_mode="warm", seasonal_weight=0.0, save_details=False):
    records = []
    recommended_catalog = set()
    discount = 1.0 / np.log2(np.arange(2, K + 2))

    if catalog_mode == "warm":
        n_candidates = n_warm_items
        item_offset = 0
        mask_train_history = True
        seasonal_slice = seasonal_features_all[:, :n_warm_items]
    elif catalog_mode == "cold":
        n_candidates = n_cold_items
        item_offset = n_warm_items
        mask_train_history = False
        seasonal_slice = seasonal_features_all[:, n_warm_items:]
    elif catalog_mode == "combined":
        n_candidates = n_all_items
        item_offset = 0
        mask_train_history = True
        seasonal_slice = seasonal_features_all
    else:
        raise ValueError(f"Unsupported catalog_mode: {catalog_mode}")

    for month, group in queries_df.groupby("month", sort=True):
        user_array = group["user_index"].to_numpy(dtype=np.int32)
        relevant_arrays = group["relevant_items"].tolist()
        for offset in range(0, len(group), SCORE_BATCH_SIZE):
            batch_users = user_array[offset:offset + SCORE_BATCH_SIZE]
            scores = score_fn(batch_users, catalog_mode).astype(np.float32, copy=True)

            if seasonal_weight > 0.0:
                scores = standardized_rows(scores)
                scores += float(seasonal_weight) * seasonal_slice[int(month) - 1]

            if mask_train_history:
                for local_idx, u_idx in enumerate(batch_users):
                    start, stop = train_counts.indptr[u_idx:u_idx + 2]
                    scores[local_idx, train_counts.indices[start:stop]] = -np.inf

            take = min(int(K), n_candidates)
            candidates = np.argpartition(scores, n_candidates - take, axis=1)[:, -take:]
            values = np.take_along_axis(scores, candidates, axis=1)
            order = np.lexsort((candidates, -values), axis=1)
            candidates = np.take_along_axis(candidates, order, axis=1) + item_offset

            targets = relevant_arrays[offset:offset + SCORE_BATCH_SIZE]
            for u_idx, pred, target in zip(batch_users, candidates, targets):
                target = np.asarray(target, dtype=np.int32)
                hits = np.isin(pred, target)
                hit_count = int(hits.sum())
                ideal_dcg = discount[:min(len(target), K)].sum()
                records.append({
                    "month": int(month), "user_index": int(u_idx), "relevant_songs": len(target),
                    f"precision@{K}": hit_count / K, f"recall@{K}": hit_count / len(target),
                    f"ndcg@{K}": float((hits * discount).sum() / ideal_dcg),
                    f"hitrate@{K}": float(hit_count > 0),
                })
                recommended_catalog.update(pred.tolist())

    detail = pd.DataFrame(records)
    overall = {
        "label": label, "catalog_mode": catalog_mode,
        **detail[METRIC_COLUMNS].mean().to_dict(),
        "queries": len(detail), "users": int(detail["user_index"].nunique()),
        "catalog_coverage": len(recommended_catalog) / n_candidates,
    }
    user_macro = detail.groupby("user_index")[METRIC_COLUMNS].mean().mean()
    overall.update({f"user_macro_{k}": float(v) for k, v in user_macro.items()})
    if save_details:
        detail.to_parquet(REPORT_DIR / f"{label}_{catalog_mode}_query_metrics.parquet", index=False)
        (REPORT_DIR / f"{label}_{catalog_mode}_metrics.json").write_text(json.dumps(overall, indent=2))
    return overall


## 9. Model 1: Multi-Interest Item-$k$NN Content Similarity (`ItemKNNContentModel`)

Item-$k$NN content similarity with hubness and popularity calibration.
https://dl.acm.org/doi/10.1145/963770.963776

Taking the content feature embedding for each song, compute its cosine similarity against every other song, keeping only the top M. During inference, use the user's listening history to compute the similar songs for every song the user listened to, aggregating them with popularity consideration and selecting top k.

In [12]:
class ItemKNNContentModel:
    def __init__(self, top_k_neighbors=100, similarity_power=1.5, pop_exponent=0.35):
        self.top_k_neighbors = int(top_k_neighbors)
        self.similarity_power = float(similarity_power)
        self.pop_exponent = float(pop_exponent)
        self.sim_warm_to_all = None
        self.item_pop_prior = None

    def fit(self, song_features, n_warm_items, batch_size=2048):
        warm_feats = song_features[:n_warm_items]
        n_all = song_features.shape[0]
        top_m = min(self.top_k_neighbors, n_warm_items)

        idf_weights = 1.0 / np.power(np.maximum(global_popularity_warm, 1.0), 0.15)
        idf_weights = (idf_weights / idf_weights.mean()).astype(np.float32)

        row_indices, col_indices, values = [], [], []
        for start in tqdm(range(0, n_all, batch_size), desc=f"Build Item-kNN (M={top_m}, g={self.similarity_power}, b={self.pop_exponent})", leave=False):
            stop = min(start + batch_size, n_all)
            cand_feats = song_features[start:stop]
            sims = cand_feats @ warm_feats.T
            for local_col in range(stop - start):
                global_item = start + local_col
                if global_item < n_warm_items:
                    sims[local_col, global_item] = 0.0

            top_idx = np.argpartition(sims, n_warm_items - top_m, axis=1)[:, -top_m:]
            top_val = np.take_along_axis(sims, top_idx, axis=1)
            top_val = np.maximum(top_val, 0.0)
            if self.similarity_power != 1.0:
                top_val = np.power(top_val, self.similarity_power)
            top_val = top_val * idf_weights[top_idx]

            batch_cols = np.repeat(np.arange(start, stop, dtype=np.int32), top_m)
            row_indices.append(top_idx.reshape(-1).astype(np.int32))
            col_indices.append(batch_cols)
            values.append(top_val.reshape(-1).astype(np.float32))

        rows = np.concatenate(row_indices)
        cols = np.concatenate(col_indices)
        vals = np.concatenate(values)
        mask = vals > 1e-7
        self.sim_warm_to_all = sparse.csr_matrix(
            (vals[mask], (rows[mask], cols[mask])), shape=(n_warm_items, n_all), dtype=np.float32
        );
        warm_prior = np.power(np.log1p(np.maximum(global_popularity_warm, 1.0)), self.pop_exponent).astype(np.float32)
        warm_prior = warm_prior / np.mean(warm_prior)
        self.item_pop_prior = np.ones(n_all, dtype=np.float32)
        self.item_pop_prior[:n_warm_items] = warm_prior
        return self

    def score_users(self, user_indices, catalog_mode="warm"):
        user_conf_batch = train_confidence[user_indices]
        if catalog_mode == "warm":
            sim_mat = self.sim_warm_to_all[:, :n_warm_items]
            prior = self.item_pop_prior[:n_warm_items]
        elif catalog_mode == "cold":
            sim_mat = self.sim_warm_to_all[:, n_warm_items:]
            prior = self.item_pop_prior[n_warm_items:]
        elif catalog_mode == "combined":
            sim_mat = self.sim_warm_to_all
            prior = self.item_pop_prior
        else:
            raise ValueError(catalog_mode)
        dense_scores = (user_conf_batch @ sim_mat).toarray()
        return dense_scores * prior[None, :]

def centroid_score_fn(user_indices, catalog_mode="warm"):
    u_vecs = user_content_profiles[user_indices]
    if catalog_mode == "warm":
        i_vecs = song_features[:n_warm_items]
    elif catalog_mode == "cold":
        i_vecs = song_features[n_warm_items:]
    else:
        i_vecs = song_features
    return u_vecs @ i_vecs.T


## 10. Model 2: Two-Tower Neural Network (`TwoTowerContentModel`)

Debiased Two-Tower Neural Network trained with uniform catalog negative sampling.

Learns user and item embeddings such that dot(user, item) produces accurate scores. User and Item MLPs both use a 2 layer network: Linear -> layer norm -> GELU -> Linear

Instead of using in-batch negative sampling (InfoNCE) where every positive item in a training batch is used as a negative example for every other user in the batch (leading to overpenalization of popular songs), we do uniform catalog negative sampling, drawing 1024 random uniform negatives from all items.

**TODO**: Currently two tower results for warm songs seem roughly on par with the collaborative filtering approach, but during training loss does not decrease as much as hoped. We need to look into whether this model is underfitting

In [13]:
class TwoTowerContentNet(nn.Module):
    def __init__(self, input_dim, n_users, hidden_dim=256, embed_dim=128, dropout=0.15):
        super().__init__()
        self.user_mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, embed_dim),
        )
        self.user_id_residual = nn.Embedding(n_users, embed_dim)
        nn.init.normal_(self.user_id_residual.weight, std=0.05)
        self.user_id_drop = nn.Dropout(p=0.2)

        self.item_mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, embed_dim),
        )
        self.input_proj = nn.Linear(input_dim, embed_dim, bias=False)

    def encode_users(self, user_profile_tensor, user_idx_tensor=None):
        u_emb = self.user_mlp(user_profile_tensor) + self.input_proj(user_profile_tensor)
        if user_idx_tensor is not None:
            u_emb = u_emb + self.user_id_drop(self.user_id_residual(user_idx_tensor))
        return F.normalize(u_emb, p=2, dim=-1)

    def encode_items(self, item_feature_tensor):
        i_emb = self.item_mlp(item_feature_tensor) + self.input_proj(item_feature_tensor)
        return F.normalize(i_emb, p=2, dim=-1)

def sample_training_pairs_for_epoch(train_confidence_csr, user_sample_fraction=1.0, max_pairs_per_user=100, seed=42):
    rng = np.random.default_rng(seed)
    all_users = np.arange(train_confidence_csr.shape[0], dtype=np.int32)
    if user_sample_fraction < 1.0:
        n_keep = max(1, int(len(all_users) * user_sample_fraction))
        selected_users = np.sort(rng.choice(all_users, size=n_keep, replace=False))
    else:
        selected_users = all_users

    u_list, i_list, w_list = [], [], []
    for u in selected_users:
        start, stop = train_confidence_csr.indptr[u:u + 2]
        items = train_confidence_csr.indices[start:stop]
        weights = train_confidence_csr.data[start:stop]
        if max_pairs_per_user is not None and len(items) > max_pairs_per_user:
            probs = weights / weights.sum()
            chosen = rng.choice(len(items), size=max_pairs_per_user, replace=False, p=probs)
            items = items[chosen]
            weights = weights[chosen]
        u_list.append(np.full(len(items), u, dtype=np.int32))
        i_list.append(items.astype(np.int32))
        w_list.append(weights.astype(np.float32))

    return np.concatenate(u_list), np.concatenate(i_list), np.concatenate(w_list)

def train_two_tower_model(params, epochs=15, seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    input_dim = song_features.shape[1]

    net = TwoTowerContentNet(
        input_dim=input_dim, n_users=n_users,
        hidden_dim=int(params["hidden_dim"]), embed_dim=int(params["embed_dim"]),
        dropout=float(params["dropout"]),
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(net.parameters(), lr=float(params["lr"]), weight_decay=1e-4)
    temperature = float(params["temperature"])
    batch_size = int(params["batch_size"])
    n_uniform_negatives = 1024

    user_profiles_t = torch.from_numpy(user_content_profiles).to(DEVICE)
    warm_feats_t = torch.from_numpy(song_features[:n_warm_items]).to(DEVICE)

    history = []
    rng = np.random.default_rng(seed)

    for epoch in tqdm(range(1, epochs + 1), desc=f"Train Debiased Two-Tower (temp={temperature})", leave=False):
        t0 = time.time()
        net.train()
        u_arr, i_arr, w_arr = sample_training_pairs_for_epoch(
            train_confidence, user_sample_fraction=1.0, max_pairs_per_user=None, seed=seed + epoch
        )
        perm = rng.permutation(len(u_arr))
        u_arr, i_arr, w_arr = u_arr[perm], i_arr[perm], w_arr[perm]

        epoch_loss = 0.0
        n_batches = 0
        for offset in range(0, len(u_arr), batch_size):
            b_u = torch.from_numpy(u_arr[offset:offset + batch_size]).long().to(DEVICE)
            b_i = torch.from_numpy(i_arr[offset:offset + batch_size]).long().to(DEVICE)
            b_w = torch.from_numpy(w_arr[offset:offset + batch_size]).float().to(DEVICE)

            u_emb = net.encode_users(user_profiles_t[b_u], b_u)
            pos_i_emb = net.encode_items(warm_feats_t[b_i])
            pos_logits = (u_emb * pos_i_emb).sum(dim=-1, keepdim=True) / temperature

            neg_idx = torch.randint(0, n_warm_items, (n_uniform_negatives,), device=DEVICE)
            neg_i_emb = net.encode_items(warm_feats_t[neg_idx])
            neg_logits = (u_emb @ neg_i_emb.T) / temperature

            collision_mask = b_i.unsqueeze(1) == neg_idx.unsqueeze(0)
            neg_logits = neg_logits.masked_fill(collision_mask, -1e4)

            logits = torch.cat([pos_logits, neg_logits], dim=1)
            targets = torch.zeros(len(b_u), dtype=torch.long, device=DEVICE)

            per_sample_loss = F.cross_entropy(logits, targets, reduction="none")
            norm_w = b_w / b_w.mean().clamp(min=1e-6)
            loss = (per_sample_loss * norm_w).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += float(loss.item())
            n_batches += 1

        history.append({"epoch": epoch, "loss": epoch_loss / max(1, n_batches), "pairs_trained": len(u_arr), "seconds": round(time.time() - t0, 2)})

    net.eval()
    with torch.no_grad():
        all_u_idx = torch.arange(n_users, device=DEVICE)
        all_user_embs = net.encode_users(user_profiles_t, all_u_idx).cpu().numpy().astype(np.float32)
        item_embs_list = []
        for start in range(0, n_all_items, 4096):
            batch_feats = torch.from_numpy(song_features[start:start + 4096]).to(DEVICE)
            item_embs_list.append(net.encode_items(batch_feats).cpu().numpy().astype(np.float32))
        all_item_embs = np.concatenate(item_embs_list, axis=0)

    return net, all_user_embs, all_item_embs, pd.DataFrame(history)


## 11. Validation Grid Search (`2018` Holdout)

Trains and evaluates Item-$k$NN and Two-Tower configurations on the 2018 Validation split across `warm`, `cold`, and `combined` discovery.


In [14]:
val_data = target_queries("val", use_full_cohort=False)
display(val_data["coverage"])

validation_rows = []
for s_weight in SEASONAL_WEIGHTS:
    for mode in ("warm", "cold", "combined"):
        res = evaluate_scorer(centroid_score_fn, val_data[mode], label=f"val_centroid_season_{s_weight:g}", catalog_mode=mode, seasonal_weight=s_weight)
        validation_rows.append({"family": "centroid_baseline", "config_id": "centroid", "seasonal_weight": float(s_weight), **res})

best_knn_score = -np.inf
best_knn_bundle = None
for idx, knn_params in enumerate(ITEM_KNN_GRID):
    config_id = f"item_knn_{idx:02d}"
    print("Fitting Item-kNN:", config_id, knn_params)
    knn_model = ItemKNNContentModel(**knn_params).fit(song_features, n_warm_items)
    for s_weight in SEASONAL_WEIGHTS:
        mode_results = {}
        for mode in ("warm", "cold", "combined"):
            res = evaluate_scorer(knn_model.score_users, val_data[mode], label=f"val_{config_id}_season_{s_weight:g}", catalog_mode=mode, seasonal_weight=s_weight)
            mode_results[mode] = res
            validation_rows.append({"family": "item_knn", "config_id": config_id, "seasonal_weight": float(s_weight), **knn_params, **res})
        if mode_results["combined"][f"ndcg@{K}"] > best_knn_score:
            best_knn_score = float(mode_results["combined"][f"ndcg@{K}"])
            best_knn_bundle = {"config_id": config_id, "params": knn_params, "seasonal_weight": float(s_weight), "model": knn_model}

best_tt_score = -np.inf
best_tt_bundle = None
for idx, tt_params in enumerate(TWO_TOWER_GRID):
    config_id = f"two_tower_{idx:02d}"
    print("Training Two-Tower:", config_id, tt_params)
    net, u_embs, i_embs, history_df = train_two_tower_model(tt_params, epochs=15, seed=SEED)
    history_df.to_csv(REPORT_DIR / f"{config_id}_training_history.csv", index=False)

    def make_tt_scorer(user_embeddings, item_embeddings):
        def _score(user_indices, catalog_mode="warm"):
            u_b = user_embeddings[user_indices]
            if catalog_mode == "warm": i_b = item_embeddings[:n_warm_items]
            elif catalog_mode == "cold": i_b = item_embeddings[n_warm_items:]
            else: i_b = item_embeddings
            return u_b @ i_b.T
        return _score

    tt_scorer = make_tt_scorer(u_embs, i_embs)
    for s_weight in SEASONAL_WEIGHTS:
        mode_results = {}
        for mode in ("warm", "cold", "combined"):
            res = evaluate_scorer(tt_scorer, val_data[mode], label=f"val_{config_id}_season_{s_weight:g}", catalog_mode=mode, seasonal_weight=s_weight)
            mode_results[mode] = res
            validation_rows.append({"family": "two_tower", "config_id": config_id, "seasonal_weight": float(s_weight), **tt_params, **res})
        if mode_results["combined"][f"ndcg@{K}"] > best_tt_score:
            best_tt_score = float(mode_results["combined"][f"ndcg@{K}"])
            best_tt_bundle = {"config_id": config_id, "params": tt_params, "seasonal_weight": float(s_weight), "user_embs": u_embs, "item_embs": i_embs, "state_dict": net.state_dict()}

val_results_df = pd.DataFrame(validation_rows)
val_results_df.to_csv(REPORT_DIR / "validation_cbf_search.csv", index=False)
display(val_results_df[val_results_df["catalog_mode"] == "combined"].sort_values(f"ndcg@{K}", ascending=False)[["family", "config_id", "seasonal_weight", f"precision@{K}", f"recall@{K}", f"ndcg@{K}", f"hitrate@{K}", "catalog_coverage"]])


,month,all_user_month_song_labels,all_active_users,cold_user_labels,cold_item_labels_among_known_users,warm_previously_seen_labels,warm_discovery_labels,eval_warm_discovery_labels,eval_cold_discovery_labels
0,1,84134,1637,80,3092,57562,23400,9497,1141
1,2,74193,1599,78,5079,47538,21498,8025,1739
2,3,79389,1605,235,6198,50000,22956,8796,2033
3,4,77296,1566,133,7738,46546,22879,8435,2748
4,5,70436,1541,178,8701,40826,20731,7482,2739
5,6,70727,1375,267,9648,39672,21140,7738,3117
6,7,76303,1359,299,10562,41112,24330,9384,3430
7,8,80922,1359,342,11419,42578,26583,9501,3868
8,9,81446,1356,449,14304,41375,25318,9264,4860
9,10,84306,1374,530,16070,41493,26213,9593,5340


Fitting Item-kNN: item_knn_00 {'top_k_neighbors': 100, 'similarity_power': 1.5, 'pop_exponent': 0.0}


Build Item-kNN (M=100, g=1.5, b=0.0):   0%|          | 0/28 [00:00<?, ?it/s]

Fitting Item-kNN: item_knn_01 {'top_k_neighbors': 100, 'similarity_power': 1.5, 'pop_exponent': 0.35}


Build Item-kNN (M=100, g=1.5, b=0.35):   0%|          | 0/28 [00:00<?, ?it/s]

...

Training Two-Tower: two_tower_00 {'embed_dim': 128, 'hidden_dim': 256, 'dropout': 0.15, 'temperature': 0.07, 'lr': 0.001, 'batch_size': 512}


Train Debiased Two-Tower (temp=0.07):   0%|          | 0/15 [00:00<?, ?it/s]

Training Two-Tower: two_tower_01 {'embed_dim': 128, 'hidden_dim': 256, 'dropout': 0.2, 'temperature': 0.05, 'lr': 0.001, 'batch_size': 512}


Train Debiased Two-Tower (temp=0.05):   0%|          | 0/15 [00:00<?, ?it/s]

,family,config_id,seasonal_weight,precision@20,recall@20,ndcg@20,hitrate@20,catalog_coverage
35,two_tower,two_tower_01,0.25,0.017702,0.017262,0.022789,0.224357,0.193559
32,two_tower,two_tower_01,0.00,0.017804,0.017023,0.022291,0.233276,0.136222
29,two_tower,two_tower_00,0.25,0.016878,0.016599,0.021223,0.219726,0.182005
26,two_tower,two_tower_00,0.00,0.016621,0.016619,0.020550,0.220926,0.131027
14,item_knn,item_knn_01,0.00,0.010695,0.010696,0.013912,0.125729,0.150641
17,item_knn,item_knn_01,0.25,0.010480,0.010460,0.013833,0.123328,0.166117
23,item_knn,item_knn_02,0.25,0.010352,0.010167,0.012516,0.130703,0.147829
20,item_knn,item_knn_02,0.00,0.010223,0.010185,0.012383,0.131732,0.135273
11,item_knn,item_knn_00,0.25,0.009194,0.009277,0.012258,0.112521,0.177957
8,item_knn,item_knn_00,0.00,0.009160,0.009288,0.012067,0.113208,0.158505


## 12. Package Locked Content-Based Models & Final Untouched `2019` Test Evaluation

Saves locked models and evaluates on the full 2,000-user evaluation cohort in the untouched 2019 Test split.


In [15]:
sparse.save_npz(BUNDLE_DIR / "best_item_knn_sim.npz", best_knn_bundle["model"].sim_warm_to_all)
np.savez_compressed(BUNDLE_DIR / "best_two_tower_embeddings.npz", user_embs=best_tt_bundle["user_embs"], item_embs=best_tt_bundle["item_embs"])
torch.save(best_tt_bundle["state_dict"], BUNDLE_DIR / "best_two_tower_weights.pt")

selection_metadata = {
    "best_item_knn": {k: v for k, v in best_knn_bundle.items() if k != "model"},
    "best_two_tower": {k: v for k, v in best_tt_bundle.items() if k not in ("user_embs", "item_embs", "state_dict")},
}
(BUNDLE_DIR / "cbf_selection.json").write_text(json.dumps(selection_metadata, indent=2))

test_data = target_queries("test", use_full_cohort=True)
locked_tt_u = best_tt_bundle["user_embs"]
locked_tt_i = best_tt_bundle["item_embs"]

def locked_tt_score_fn(user_indices, catalog_mode="warm"):
    u_b = locked_tt_u[user_indices]
    if catalog_mode == "warm": i_b = locked_tt_i[:n_warm_items]
    elif catalog_mode == "cold": i_b = locked_tt_i[n_warm_items:]
    else: i_b = locked_tt_i
    return u_b @ i_b.T

test_rows = []
for mode in ("warm", "cold", "combined"):
    test_rows.append(evaluate_scorer(best_knn_bundle["model"].score_users, test_data[mode], label="test_item_knn_selected", catalog_mode=mode, seasonal_weight=best_knn_bundle["seasonal_weight"], save_details=True))
    test_rows.append(evaluate_scorer(locked_tt_score_fn, test_data[mode], label="test_two_tower_selected", catalog_mode=mode, seasonal_weight=best_tt_bundle["seasonal_weight"], save_details=True))
    test_rows.append(evaluate_scorer(centroid_score_fn, test_data[mode], label="test_centroid_baseline", catalog_mode=mode, seasonal_weight=0.0, save_details=True))

test_results_df = pd.DataFrame(test_rows)
test_results_df.to_csv(REPORT_DIR / "test_cbf_summary.csv", index=False)
display(test_results_df)


,label,catalog_mode,precision@20,recall@20,ndcg@20,hitrate@20,queries,users,catalog_coverage,user_macro_precision@20,user_macro_recall@20,user_macro_ndcg@20,user_macro_hitrate@20
0,test_item_knn_selected,warm,0.010345,0.010617,0.013015,0.129706,5312,580,0.164627,0.008674,0.009824,0.011227,0.109763
1,test_two_tower_selected,warm,0.018119,0.017556,0.022768,0.233998,5312,580,0.249558,0.015830,0.017900,0.020786,0.204671
2,test_centroid_baseline,warm,0.006099,0.005798,0.007013,0.087161,5312,580,0.133074,0.005222,0.005612,0.006192,0.074301
3,test_item_knn_selected,cold,0.019473,0.041798,0.033141,0.196484,4550,528,0.260826,0.016652,0.043405,0.032561,0.168365
4,test_two_tower_selected,cold,0.021165,0.045502,0.036421,0.249451,4550,528,0.249321,0.018012,0.050071,0.035466,0.217329
5,test_centroid_baseline,cold,0.013341,0.029469,0.021320,0.158462,4550,528,0.191725,0.011505,0.030734,0.020532,0.140038
6,test_item_knn_selected,combined,0.012674,0.011010,0.015164,0.148115,5543,592,0.127606,0.010930,0.011922,0.013725,0.128263
7,test_two_tower_selected,combined,0.022244,0.017557,0.026788,0.271333,5543,592,0.176166,0.019283,0.019883,0.024875,0.238336
8,test_centroid_baseline,combined,0.006964,0.005034,0.007994,0.096338,5543,592,0.104643,0.005904,0.004603,0.006915,0.081387


## 13. Cold-Start Song Inference Demo

Demonstrates embedding a brand-new cold song and querying its nearest neighbors.


In [16]:
if n_cold_items > 0:
    sample_cold_global_idx = n_warm_items
    sample_cold_id = item_ids[sample_cold_global_idx]
    cold_vec = locked_tt_i[sample_cold_global_idx]
    cos_sims = locked_tt_i @ cold_vec
    top_nn = np.argsort(-cos_sims)[1:6]
    nn_rows = []
    for rank, idx in enumerate(top_nn, start=1):
        nn_rows.append({"rank": rank, "neighbor_item_id": item_ids[idx], "is_warm_in_train": bool(item_is_warm[idx]), "two_tower_cosine_sim": round(float(cos_sims[idx]), 4)})
        print(f"Cold-start track '{sample_cold_id}' -> Top-5 Two-Tower Neighbors:")
    display(pd.DataFrame(nn_rows))


Cold-start track '00hFmpewuNvc4dRL' -> Top-5 Two-Tower Neighbors:
Cold-start track '00hFmpewuNvc4dRL' -> Top-5 Two-Tower Neighbors:
Cold-start track '00hFmpewuNvc4dRL' -> Top-5 Two-Tower Neighbors:
Cold-start track '00hFmpewuNvc4dRL' -> Top-5 Two-Tower Neighbors:
Cold-start track '00hFmpewuNvc4dRL' -> Top-5 Two-Tower Neighbors:


,rank,neighbor_item_id,is_warm_in_train,two_tower_cosine_sim
0,1,Y29XLA6gDZFiKeOA,True,0.6906
1,2,OUb23fwEUhwC3Uec,True,0.6685
2,3,zTdC3xWJNtaYyjV2,True,0.6629
3,4,jxYmnXCxlmIdiZhh,True,0.6594
4,5,E3iQOe4M979du9C8,True,0.6511
